# 📦 Stage 0: Data Preparation & Dataset Mixing
> Download, combine, and prepare all training datasets for the Code LLM Forge pipeline.

**What this notebook does:**
1. Downloads all required datasets from HuggingFace
2. Formats them into a unified chat template (Qwen2.5-Coder format)
3. Mixes datasets according to our research-backed ratios
4. Saves prepared datasets for Stage 1, 2, and 3 training

**Requirements:** Any GPU tier (data prep is CPU-bound). ~20GB disk space.


In [ ]:
# Install dependencies
!pip install -q datasets transformers huggingface_hub tqdm

In [ ]:
import os
import json
import random
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm.auto import tqdm

# Set seed for reproducibility
random.seed(42)

# Output directory
os.makedirs("prepared_data", exist_ok=True)
print("✅ Setup complete")

## 1. Download Datasets

### Stage 1 Datasets (Code + Reasoning)
| Dataset | Samples | Purpose |
|---------|---------|---------|
| CodeFeedback-Filtered-Instruction | 156K | High-quality code instructions |
| Magicoder-OSS-Instruct-75K | 75K | Real OSS-grounded code |
| Magicoder-Evol-Instruct-110K | 110K | Evolved complexity tasks |


In [ ]:
# Download Stage 1 datasets
print("📥 Downloading Stage 1 datasets...")

# CodeFeedback-Filtered-Instruction (156K, complexity-filtered from 4 sources)
ds_codefeedback = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train")
print(f"  ✅ CodeFeedback: {len(ds_codefeedback):,} samples")

# Magicoder-OSS-Instruct-75K (real open-source grounded)
ds_magicoder_oss = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
print(f"  ✅ Magicoder-OSS: {len(ds_magicoder_oss):,} samples")

# Magicoder-Evol-Instruct-110K (evolved complexity)
ds_magicoder_evol = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train")
print(f"  ✅ Magicoder-Evol: {len(ds_magicoder_evol):,} samples")

print(f"\n📊 Stage 1 total raw: {len(ds_codefeedback) + len(ds_magicoder_oss) + len(ds_magicoder_evol):,} samples")

In [ ]:
# Download Stage 2 datasets (Tool Calling)
print("📥 Downloading Stage 2 datasets...")

# Hermes Function-Calling V1 (multi-turn, best quality)
ds_hermes_fc = load_dataset("NousResearch/hermes-function-calling-v1", split="train")
print(f"  ✅ Hermes FC: {len(ds_hermes_fc):,} samples")

# Glaive Function-Calling V2 (113K examples)
ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(f"  ✅ Glaive FC: {len(ds_glaive):,} samples")

print(f"\n📊 Stage 2 total raw: {len(ds_hermes_fc) + len(ds_glaive):,} samples")

## 2. Format into Qwen2.5-Coder Chat Template

All datasets must be converted to a unified format that matches the Qwen2.5-Coder-7B-Instruct chat template:

```
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful coding assistant.

# Tools
You may call one or more functions...
<tools>
[function schemas]
</tools>
<|im_end|>
<|im_start|>user
[user message]<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "function_name", "arguments": {"key": "value"}}
</tool_call><|im_end|>
```


In [ ]:
# Unified formatting functions

SYSTEM_PROMPT_CODE = "You are a helpful coding assistant. Think step-by-step and provide clear, well-documented code."

SYSTEM_PROMPT_TOOL = """You are a helpful coding assistant with tool-calling capabilities.

# Tools

You may call one or more functions to assist with the user query.

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{{"name": <function-name>, "arguments": <args-json-object>}}
</tool_call>"""

def format_codefeedback(example):
    """Format CodeFeedback-Filtered-Instruction into chat messages."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CODE},
        {"role": "user", "content": example.get("query", example.get("instruction", ""))},
        {"role": "assistant", "content": example.get("answer", example.get("response", ""))}
    ]
    return {"messages": messages, "source": "codefeedback", "task_type": "code_generation"}

def format_magicoder(example):
    """Format Magicoder datasets into chat messages."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CODE},
        {"role": "user", "content": example.get("problem", example.get("instruction", ""))},
        {"role": "assistant", "content": example.get("solution", example.get("response", ""))}
    ]
    return {"messages": messages, "source": "magicoder", "task_type": "code_generation"}

def format_hermes_fc(example):
    """Format Hermes function-calling into chat messages.
    Hermes uses ShareGPT format with 'conversations' field.
    """
    conversations = example.get("conversations", [])
    messages = []
    for turn in conversations:
        role_map = {"system": "system", "human": "user", "gpt": "assistant", "tool": "tool"}
        role = role_map.get(turn.get("from", ""), turn.get("from", ""))
        messages.append({"role": role, "content": turn.get("value", "")})

    if not any(m["role"] == "system" for m in messages):
        messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT_TOOL})

    return {"messages": messages, "source": "hermes_fc", "task_type": "tool_calling"}

def format_glaive(example):
    """Format Glaive function-calling into chat messages."""
    # Glaive uses a specific format with system/user/assistant turns
    chat_text = example.get("chat", "")
    system_text = example.get("system", SYSTEM_PROMPT_TOOL)

    messages = [{"role": "system", "content": system_text}]

    # Parse the chat field which contains USER: ... ASSISTANT: ... patterns
    parts = chat_text.split("USER: ")
    for part in parts[1:]:  # Skip empty first split
        if "ASSISTANT: " in part:
            user_msg, assistant_msg = part.split("ASSISTANT: ", 1)
            messages.append({"role": "user", "content": user_msg.strip()})
            messages.append({"role": "assistant", "content": assistant_msg.strip()})
        else:
            messages.append({"role": "user", "content": part.strip()})

    return {"messages": messages, "source": "glaive_fc", "task_type": "tool_calling"}

print("✅ Formatting functions defined")

In [ ]:
# Apply formatting to all datasets
print("🔄 Formatting Stage 1 datasets...")

stage1_samples = []

# CodeFeedback
for ex in tqdm(ds_codefeedback, desc="CodeFeedback"):
    try:
        formatted = format_codefeedback(ex)
        if formatted["messages"][1]["content"] and formatted["messages"][2]["content"]:
            stage1_samples.append(formatted)
    except Exception:
        continue

# Magicoder OSS
for ex in tqdm(ds_magicoder_oss, desc="Magicoder-OSS"):
    try:
        formatted = format_magicoder(ex)
        if formatted["messages"][1]["content"] and formatted["messages"][2]["content"]:
            stage1_samples.append(formatted)
    except Exception:
        continue

# Magicoder Evol
for ex in tqdm(ds_magicoder_evol, desc="Magicoder-Evol"):
    try:
        formatted = format_magicoder(ex)
        if formatted["messages"][1]["content"] and formatted["messages"][2]["content"]:
            stage1_samples.append(formatted)
    except Exception:
        continue

print(f"\n📊 Stage 1 formatted: {len(stage1_samples):,} samples")

# Shuffle with seed
random.shuffle(stage1_samples)

# Save
with open("prepared_data/stage1_code_reasoning.jsonl", "w") as f:
    for sample in stage1_samples:
        f.write(json.dumps(sample) + "\n")

print(f"💾 Saved to prepared_data/stage1_code_reasoning.jsonl")

In [ ]:
# Format Stage 2 datasets
print("🔄 Formatting Stage 2 datasets...")

stage2_samples = []

# Hermes FC (60% of tool calling allocation)
for ex in tqdm(ds_hermes_fc, desc="Hermes FC"):
    try:
        formatted = format_hermes_fc(ex)
        if len(formatted["messages"]) >= 2:
            stage2_samples.append(formatted)
    except Exception:
        continue

# Glaive FC (25% of tool calling allocation)
glaive_limit = int(len(stage2_samples) * 0.42)  # To achieve ~25% of total
for i, ex in enumerate(tqdm(ds_glaive, desc="Glaive FC")):
    if i >= glaive_limit:
        break
    try:
        formatted = format_glaive(ex)
        if len(formatted["messages"]) >= 3:
            stage2_samples.append(formatted)
    except Exception:
        continue

print(f"\n📊 Stage 2 formatted: {len(stage2_samples):,} samples")

random.shuffle(stage2_samples)

with open("prepared_data/stage2_tool_calling.jsonl", "w") as f:
    for sample in stage2_samples:
        f.write(json.dumps(sample) + "\n")

print(f"💾 Saved to prepared_data/stage2_tool_calling.jsonl")

In [ ]:
# Summary
import os

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

print("=" * 60)
print("📊 DATASET PREPARATION SUMMARY")
print("=" * 60)
print(f"Stage 1 (Code + Reasoning): {len(stage1_samples):>10,} samples  ({file_size_mb('prepared_data/stage1_code_reasoning.jsonl'):.1f} MB)")
print(f"Stage 2 (Tool Calling):     {len(stage2_samples):>10,} samples  ({file_size_mb('prepared_data/stage2_tool_calling.jsonl'):.1f} MB)")
print(f"{'─' * 60}")
print(f"Total:                      {len(stage1_samples) + len(stage2_samples):>10,} samples")
print()
print("✅ Ready for training! Proceed to notebook 02.")